# Avaliação de Aceitação de Tecnologia (TAM) — Dashboard SISCAN

**Objetivo:** documentar de maneira rastreável os cálculos empregados na avaliação TAM, com explicações das fórmulas, exemplos retirados das respostas elegíveis e geração dos gráficos/tabelas em PNG para o artigo.

**Fonte:** exportação original do Google Forms, novembro de 2026. **Método:** análise quantitativa descritiva; não são feitas inferências populacionais. O notebook é independente das camadas Bronze, Silver e Gold e do aplicativo Streamlit.

**Roteiro:** (1) selecionar a amostra → (2) codificar a escala → (3) calcular cada indicador da tabela → (4) conferir resultados → (5) apresentar itens/funções/perfil → (6) exportar os arquivos PNG.

## 1. Carregamento da base e seleção dos participantes

A exportação contém 43 respostas. Uma pessoa não consentiu: sua resposta é removida. Outra consentiu, mas marcou que não conseguiu acessar/explorar a plataforma: entra na descrição das respostas consentidas, mas não nas avaliações de experiência de uso.

**Amostra TAM:** 41 participantes que consentiram e declararam ter explorado a plataforma. A base original permanece inalterada. Todos os cálculos abaixo utilizam essa base, salvo indicação explícita.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

candidatos = [Path.cwd() / "avaliacao_tam", Path.cwd(), Path.cwd().parent, Path.cwd().parent / "avaliacao_tam"]
pasta = next((p for p in candidatos if (p / "analise_tam.py").is_file()), None)
if pasta is None:
    raise FileNotFoundError("Abra o notebook dentro de avaliacao_tam/ ou da raiz do projeto.")
sys.path.insert(0, str(pasta.resolve()))
import analise_tam as tam

bruto, consentidos, elegiveis = tam.ler_bases()
fluxo = tam.resumo_amostra(bruto, consentidos, elegiveis)
display(fluxo)
assert (len(bruto), len(consentidos), len(elegiveis)) == (43, 42, 41)

## 2. Escala Likert e mapa dos itens

As cinco alternativas dos 16 itens TAM são convertidas nos valores **1 a 5**. Todos os enunciados foram formulados positivamente; não há recodificação reversa. As questões sobre funcionalidades são analisadas separadamente: a opção **"Não utilizei / Não se aplica" não recebe pontuação** e não entra nos denominadores de média/concordância.

| Resposta | Código numérico |
|:---|---:|
| Discordo totalmente | 1 |
| Discordo parcialmente | 2 |
| Nem concordo nem discordo | 3 |
| Concordo parcialmente | 4 |
| Concordo totalmente | 5 |

| Construto | Sigla | Itens | Quantidade de itens ($k$) |
|:---|:---:|:---|---:|
| Utilidade percebida | PU | PU1 a PU5 | 5 |
| Facilidade de uso percebida | PEOU | PEOU1 a PEOU5 | 5 |
| Atitude em relação ao uso | AT | AT1 a AT3 | 3 |
| Intenção de uso | BI | BI1 a BI3 | 3 |

Para auditar os cálculos, criamos uma matriz **participantes × itens**, em que cada célula corresponde a uma resposta já transformada em número.

In [ ]:
matrizes = {}
for sigla, indices in tam.CONSTRUTOS.items():
    matrizes[sigla] = pd.DataFrame({
        f"{sigla}{j}": tam.codificar(elegiveis.iloc[:, coluna])
        for j, coluna in enumerate(indices, start=1)
    }, index=elegiveis.index)

print("Formato da matriz PU:", matrizes["PU"].shape)
display(matrizes["PU"].head(5))
print("Itens ausentes por construto:", {k: int(v.isna().sum().sum()) for k, v in matrizes.items()})

## 3. Como foi construída a tabela "Síntese dos quatro construtos TAM"

A tabela possui sete colunas: **Construto, Itens, n, Média, Mediana, Concordância e Alfa**. A seguir, cada coluna é definida e reproduzida numericamente. É importante distinguir os indicadores calculados *por participante* daqueles calculados *sobre o conjunto de respostas aos itens*.

### 3.1. Quantidade de itens ($k$) e participantes ($n$)

- **Itens ($k$):** número de afirmações que compõem o construto; por exemplo, PU = 5.
- **$n$ participantes:** número de participantes com **todos os itens daquele construto respondidos**; nesta base, $n=41$ nos quatro construtos.
- **Respostas aos itens:** $k\times n$ quando não há omissões; PU possui $5\times41=205$ respostas válidas, enquanto AT possui $3\times41=123$.

### 3.2. Escore individual de cada participante

Para um construto $c$, com $k_c$ itens, o escore da pessoa $i$ é a **média dos seus itens**:

$$
E_{ic}=\frac{1}{k_c}\sum_{j=1}^{k_c} x_{ij},
$$

em que $x_{ij}$ é a resposta da pessoa $i$ ao item $j$, de 1 a 5.

Exemplo **real e desidentificado** de PU encontrado na base: respostas **5, 5, 5, 4 e 5**. O escore é $(5+5+5+4+5)/5=4{,}80$. O exemplo não apresenta identificação ou outros campos da resposta.

In [ ]:
pu = matrizes["PU"]
# Seleciona o primeiro perfil 5/5/5/4/5 encontrado na própria base, sem expor identificação.
exemplo = pu.loc[(pu == [5, 5, 5, 4, 5]).all(axis=1)].iloc[0]
print("Respostas PU de um exemplo:", exemplo.astype(int).tolist())
print("Escore individual PU =", " + ".join(map(str, exemplo.astype(int).tolist())),
      f"/ 5 = {exemplo.mean():.2f}".replace(".", ","))
assert np.isclose(exemplo.mean(), 4.8)

escores_auditaveis = pd.DataFrame(index=elegiveis.index)
for sigla, matriz in matrizes.items():
    escores_auditaveis[sigla] = matriz.mean(axis=1).where(matriz.notna().all(axis=1))
print("Escores individuais calculados (sem nomes):")
display(escores_auditaveis.head())

### 3.3. Média do construto

A **média apresentada na tabela** é a média dos escores individuais completos dos participantes, e **não** uma classificação arbitrária de aprovação:

$$
\overline E_c=\frac{1}{n_c}\sum_{i=1}^{n_c}E_{ic}.
$$

Por exemplo, para PU, é calculada a média dos **41 escores individuais** de PU. O resultado não é o mesmo tipo de indicador que a concordância positiva (definida adiante). Se todos os participantes responderem igualmente todos os itens, a média global das respostas aos itens coincide numericamente com a média dos escores individuais; ainda assim, a unidade do cálculo permanece distinta.

### 3.4. Mediana

A **mediana** utiliza os escores individuais ordenados, não as 205 respostas brutas. Como há 41 participantes, a mediana é o **21º valor da lista ordenada** dos 41 escores. Para PU, resulta em **4,80**.

### 3.5. Desvio-padrão amostral (conferência adicional)

Embora não apareça na tabela PNG resumida, o script guarda o desvio-padrão amostral dos escores individuais:

$$
s_c=\sqrt{\frac{\sum_{i=1}^{n_c}(E_{ic}-\overline E_c)^2}{n_c-1}}.
$$

Esse valor não deve ser confundido com as variâncias dos *itens* utilizadas no alfa de Cronbach.

In [ ]:
linhas_escores = []
for sigla, serie in escores_auditaveis.items():
    validos = serie.dropna()
    linhas_escores.append({
        "Construto": sigla,
        "Itens": matrizes[sigla].shape[1],
        "n participantes": len(validos),
        "Média": validos.mean(),
        "Mediana": validos.median(),
        "Desvio-padrão amostral": validos.std(ddof=1),
    })
resumo_escores = pd.DataFrame(linhas_escores)
display(resumo_escores.round(4))
assert np.isclose(resumo_escores.loc[0, "Média"], 4.570731707317074)
assert np.isclose(resumo_escores.loc[0, "Mediana"], 4.80)

pu_escores_ordenados = escores_auditaveis["PU"].dropna().sort_values().to_numpy()
print(f"PU — mediana: 21º valor ordenado = {pu_escores_ordenados[20]:.2f}".replace(".", ","))

### 3.6. Concordância positiva — **atenção ao denominador**

Uma resposta é classificada como positiva quando recebe **4 (concordo parcialmente)** ou **5 (concordo totalmente)**. A concordância do construto considera **todas as respostas válidas de todos os itens**, e não o número de pessoas com escore alto:

$$
C_c(\%)=\frac{\sum_{j=1}^{k_c}(n_{4,j}+n_{5,j})}{\sum_{j=1}^{k_c}n_{\mathrm{válido},j}}\times100.
$$

**Exemplo com PU:**

- 41 participantes × 5 afirmações = **205 respostas válidas**.
- Somando as respostas **4 ou 5** em PU1, PU2, PU3, PU4 e PU5, encontram-se **193 respostas positivas**.
- Concordância positiva PU = **193 ÷ 205 × 100 = 94,15%**.

**Interpretação correta:** 94,15% das **respostas aos itens PU** foram concordância parcial ou total. **Não afirmar que 94,15% dos participantes aprovaram a plataforma**, pois isso seria outro indicador.

In [ ]:
linhas_concordancia = []
for sigla, matriz in matrizes.items():
    contagens = matriz.stack().value_counts().reindex(range(1, 6), fill_value=0)
    positivos = int(contagens.loc[4] + contagens.loc[5])
    n_validas = int(contagens.sum())
    linhas_concordancia.append({
        "Construto": sigla,
        "n respostas 1": int(contagens.loc[1]),
        "n respostas 2": int(contagens.loc[2]),
        "n respostas 3": int(contagens.loc[3]),
        "n respostas 4": int(contagens.loc[4]),
        "n respostas 5": int(contagens.loc[5]),
        "Positivas (4+5)": positivos,
        "Válidas (1 a 5)": n_validas,
        "Concordância (%)": 100*positivos/n_validas if n_validas else np.nan
    })
resumo_concordancia = pd.DataFrame(linhas_concordancia)
display(resumo_concordancia.round(2))
pu_conc = resumo_concordancia.loc[resumo_concordancia["Construto"] == "PU"].iloc[0]
assert (pu_conc["Positivas (4+5)"], pu_conc["Válidas (1 a 5)"]) == (193, 205)
print("PU:", f"193 / 205 × 100 = {100*193/205:.2f}%".replace(".", ","))

### 3.7. Alfa de Cronbach — cálculo completo

O **alfa de Cronbach** quantifica a consistência interna dos itens agrupados em um mesmo construto. O cálculo usa as **variâncias amostrais** dos itens e a variância amostral da **soma das pontuações por participante**:

$$
\alpha_c=\frac{k_c}{k_c-1}\left(1-\frac{\sum_{j=1}^{k_c}s_j^2}{s_T^2}\right),
$$

em que:

- $k_c$: quantidade de itens do construto;
- $s_j^2$: variância **amostral** das respostas ao item $j$, calculada entre os participantes;
- $T_i = \sum_{j=1}^{k_c} x_{ij}$: **soma** (não média) dos itens do participante $i$;
- $s_T^2$: variância **amostral** das somas $T_i$.

Empregamos denominador $n-1$ (`ddof=1`) para todas as variâncias e somente **casos completos**. O alfa não é uma porcentagem nem prova validade, unidimensionalidade ou adequação do instrumento por si só.

**Exemplo numérico de PU**, com arredondamento apenas para apresentação:

- $k=5$ itens;
- soma das cinco variâncias individuais: $\sum s_j^2 \approx 2{,}8780$;
- variância das somas das cinco respostas: $s_T^2 \approx 11{,}3280$;
- $\alpha_{PU}=\frac54(1-\frac{2{,}8780}{11{,}3280})\approx0{,}9324$ → **0,93** na tabela.

O código abaixo reproduz **item por item** as variâncias e calcula os quatro coeficientes.

In [ ]:
linhas_alpha = []
for sigla, matriz in matrizes.items():
    completos = matriz.dropna()
    k = completos.shape[1]
    var_itens = completos.var(axis=0, ddof=1)
    var_somas = completos.sum(axis=1).var(ddof=1)
    alpha = (k/(k-1))*(1-var_itens.sum()/var_somas)
    linhas_alpha.append({
        "Construto": sigla, "k": k, "n completo": len(completos),
        "Σ variâncias itens": var_itens.sum(), "Var(soma itens)": var_somas,
        "Alfa calculado": alpha,
    })
    print(f"\n{sigla}: variância amostral de cada item")
    display(var_itens.rename("Variância amostral").to_frame().round(5))
    print(f"α = ({k}/{k-1}) × (1 − {var_itens.sum():.5f}/{var_somas:.5f}) = {alpha:.5f}")
resumo_alpha = pd.DataFrame(linhas_alpha)
print("\nSíntese dos cálculos do alfa:")
display(resumo_alpha.round(5))
assert np.isclose(resumo_alpha.loc[0, "Alfa calculado"], 0.932420, atol=0.00001)

### 3.8. Montagem e verificação da tabela final

Agora reunimos **as sete colunas da tabela PNG**. Cada valor provém dos cálculos anteriores: `k` e `n` da matriz de respostas, média e mediana dos escores individuais, concordância do conjunto de itens e alfa das variâncias amostrais. O script oficial `analise_tam.py` é usado como segunda conferência para evitar divergências.

**Arredondamento:** todos os cálculos são feitos com precisão numérica completa. Apenas **na apresentação** os indicadores recebem duas casas decimais e vírgula decimal. O alfa é apresentado sem o símbolo `%`.

In [ ]:
tabela_calculada = (
    resumo_escores[["Construto", "Itens", "n participantes", "Média", "Mediana"]]
    .merge(resumo_concordancia[["Construto", "Concordância (%)"]], on="Construto")
    .merge(resumo_alpha[["Construto", "Alfa calculado"]], on="Construto")
)
tabela_calculada["Construto"] = tabela_calculada["Construto"].map(tam.NOMES)
tabela_calculada.columns = ["Construto", "Itens", "n", "Média", "Mediana", "Concordância (%)", "Alfa de Cronbach"]
display(tabela_calculada.round(4))

resumo_modulo, itens_tam, escores_modulo = tam.analise_construtos(elegiveis)
for calculado, esperado in [("Média", "Média"), ("Mediana", "Mediana"),
                            ("Concordância (%)", "Concordância positiva (%)"),
                            ("Alfa de Cronbach", "Alfa de Cronbach")]:
    assert np.allclose(tabela_calculada[calculado], resumo_modulo[esperado], equal_nan=True)
print("Conferência OK: cálculos passo a passo = cálculos usados na tabela PNG.")

## 4. Respostas aos 16 itens e desenho das figuras

A figura principal foi **redesenhada para leitura no artigo**. Ela traz quatro painéis (PU, PEOU, AT e BI), com o **percentual de concordância positiva de cada item**, sem segmentos estreitos de 1% a 5% e sem uma legenda de cinco cores. As porcentagens têm duas casas decimais.

**Limitação:** o gráfico principal não mostra separadamente as cinco categorias Likert. Para inspeção do perfil completo de discordância, neutralidade e concordância, o script também produz **quatro figuras suplementares**, cada uma com um construto em barras empilhadas de 100%. Os gráficos suplementares preservam todas as alternativas de 1 a 5.

As descrições abreviadas têm correspondência com o enunciado completo no campo `Afirmação` da tabela `03_itens_tam.csv`.

In [ ]:
itens_auditoria = itens_tam[["Código", "Afirmação", "n válido", "n(1)", "n(2)",
                               "n(3)", "n(4)", "n(5)", "Concordância positiva (%)"]]
display(itens_auditoria.round(2))
print("Exemplo PU1: (n4 + n5) / n válido =",
      f"({int(itens_tam.loc[0,'n(4)'])} + {int(itens_tam.loc[0,'n(5)'])}) / {int(itens_tam.loc[0,'n válido'])} × 100 =",
      f"{itens_tam.loc[0,'Concordância positiva (%)']:.2f}%".replace(".", ","))

## 5. Avaliação das funcionalidades (F1–F8)

As oito afirmações de funcionalidades **não são incorporadas aos escores TAM**. Para cada funcionalidade:

$$
C_F(\%)=\frac{n_4+n_5}{n_{\mathrm{válido}}}\times100,
\quad
n_{\mathrm{válido}}=n_{1..5}.
$$

As respostas "Não utilizei / Não se aplica" são contadas em outra coluna e excluídas desse denominador. Exemplo: **exportação de gráficos e tabelas** recebeu 27 avaliações válidas e 14 marcações de não utilização.

In [ ]:
funcionalidades = tam.analise_funcionalidades(elegiveis)
display(funcionalidades[["Código", "Funcionalidade", "n válido", "Não utilizou (n)",
                        "Concordância positiva (n)", "Concordância positiva (%)"]].round(2))
exportacao = funcionalidades.loc[funcionalidades["Código"] == "F7"].iloc[0]
print("Exportação: concordância =",
      f"{int(exportacao['Concordância positiva (n)'])}/{int(exportacao['n válido'])} × 100 = ",
      f"{exportacao['Concordância positiva (%)']:.2f}%".replace(".", ","),
      "| Não utilizou:", int(exportacao["Não utilizou (n)"]))

## 6. Perfil, dificuldades e comentários livres

As questões de **múltipla seleção** contabilizam cada alternativa separadamente, então os percentuais podem ultrapassar 100% quando somados. Foi detectado um caso em que a mesma resposta marcou "Não encontrei dificuldades" e uma dificuldade: para quantificar as dificuldades, prevalece a marcação da dificuldade e a inconsistência é documentada.

**Respostas abertas:** o script cria uma ficha para **codificação temática manual**; nenhuma frequência temática é atribuída sem leitura e revisão do conteúdo. Não confundir a quantidade de comentários com a quantidade de categorias identificadas.

In [ ]:
perfis = tam.perfil(elegiveis)
for nome, tabela in perfis.items():
    print("\n", nome)
    display(tabela.round(2))
dif, inconsistencias = tam.dificuldades(elegiveis)
print("\nDificuldades relatadas | inconsistências:", inconsistencias)
display(dif.round(2))
ficha = tam.abertas(elegiveis)
print("Sugestões abertas:", int(ficha["Sugestão de melhoria"].str.strip().ne("").sum()))
print("Comentários adicionais:", int(ficha["Comentário adicional"].str.strip().ne("").sum()))

## 7. Exportar imagens e tabelas (PNG, 320 dpi)

A função seguinte recria **as figuras em `figuras/`** e **as tabelas em `tabelas/`**, todas em PNG a 320 dpi, além das tabelas CSV de auditoria. O processamento dos dados não altera o CSV original.

**Arquivos indicados para o artigo:**

- `tabelas/02_indicadores_construtos.png` — tabela dos quatro construtos TAM;
- `figuras/02_itens_tam_concordancia.png` — figura principal redesenhada dos 16 itens;
- `figuras/02_likert_pu.png`, `02_likert_peou.png`, `02_likert_at.png`, `02_likert_bi.png` — distribuição Likert completa, uma figura por construto;
- `figuras/04_funcionalidades_concordancia.png` — funcionalidades.

Os PNGs contêm somente a visualização e o título interno, **sem rodapé de fonte ou observações**. Ao inserir no Overleaf, informe a fonte, as notas metodológicas necessárias e a legenda no ambiente LaTeX, evitando duplicidade no resultado final.

In [ ]:
resultado = tam.executar()
display(pd.DataFrame([resultado["metadados"]]))
for nome in ["tabelas/02_indicadores_construtos.png", "figuras/02_itens_tam_concordancia.png"]:
    imagem = pasta / nome
    assert imagem.exists(), imagem
    print("Imagem gerada:", imagem.relative_to(pasta))
    display(Image(filename=str(imagem), width=920))

## 8. Interpretação e cuidados para redigir os resultados

1. **Média** é um indicador do escore composto por participante, enquanto **concordância positiva** agrega respostas aos itens. Não trocar os denominadores.
2. **Alfa de Cronbach** é medida de consistência interna; não demonstra sozinho validade ou unidimensionalidade. Os quatro coeficientes devem ser interpretados como exploratórios, considerando a amostra de 41 pessoas.
3. O estudo descreve a percepção **dos participantes elegíveis**. Não extrapolar automaticamente os resultados para todos os profissionais da 10ª CRS.
4. Avaliações de funcionalidades com "Não utilizei" devem apresentar **$n$ válido por pergunta**.
5. **Intenção de uso** não equivale ao uso futuro observado.
6. Ao citar percentuais, utilizar **duas casas decimais** e informar a fonte e os denominadores relevantes.

Sugestão de redação para PU: *"A utilidade percebida obteve escore médio de 4,57 e mediana de 4,80. Das 205 respostas válidas registradas nos cinco itens desse construto, 193 (94,15%) corresponderam a concordância parcial ou total. O coeficiente alfa de Cronbach foi de 0,93, sugerindo consistência interna elevada no conjunto de itens, sem demonstrar isoladamente a validade do construto."*